### In this file we will see how tools can be used in LangGraph, both custom tool and built-in tools

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool

from langchain_community.tools import DuckDuckGoSearchRun

from typing import TypedDict, Annotated, Literal
import operator
from dotenv import load_dotenv

from pydantic import BaseModel, Field

load_dotenv()

True

In [8]:
# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant")

#### Creating the tools

In [9]:
# 1. search tool (This is the built-in tool)
search_tool = DuckDuckGoSearchRun(name='brave_search')  ## As we are using Groq's Llama model, that uses by default brave_search so we are naming our tool in the same name to force it go via DuckDuckGoSearch only

# 2. Calculator (This is the Customized tool )
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    ## Adding the docstring is important as LLM will read this at the time of tool triggering
    """ Perform a basic arithmatic operation on the numbers. Supported operation - add, sub, mul, div"""

    if operation == 'add':
        result = first_num+second_num
        return {"result": first_num + second_num}

    elif operation == 'sub':
        result = first_num-second_num
        return {"result": first_num - second_num}

    elif operation == 'mul':
        result = first_num*second_num
        return {"result": first_num * second_num}

    elif operation == 'div':
        if second_num == 0:
            return {"error": "Division by zero is not allowed"}
        else:
            result = first_num/second_num
            return {"result": first_num / second_num}
    else:
        return {"error": f"unsupported operation- {operation}"}
        

## Make the tool list that will be passed to langgraph
tools = [search_tool, calculator]

## Make the LLM aware about the tools
llm_with_tools = llm.bind_tools(tools)

### Langgraph portion

In [12]:
my_checkpointer = InMemorySaver()

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  ## To add the messages one after one as chat format


## Node functions
def chat_node(state: ChatState):
    """LLM node that may answer or request tool call"""
    message = state['messages']
    response = llm_with_tools.invoke(message)
    return {'messages': [response]}


## Adding the tools to the Langgraph node
tool_node = ToolNode(tools)


## Creating nodes and edges
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)  ## It will judge to end the workflow or to use any tool
graph.add_edge('tools', 'chat_node')   ## If 2 tool call is needed then it will again go back to the chat history and then trigger require tool

workflow = graph.compile(checkpointer = my_checkpointer)

In [14]:
output = workflow.invoke({'messages': [HumanMessage(content="Tell me the stock price of TSLA")]}, config={'configurable': {'thread_id': 'v1'}})
# print(output)
# print("==============================================")
print(output['messages'][-1].content)

The stock price of TSLA as of 3 March 2026 is $403.32.


In [15]:
# Making in chatbot manner
thread_id = 'Session_001'
exit_keywords = ['exit', 'stop', 'bye']

while True:
    user_msg = input("Please ask your query")
    if user_msg.lower().strip() in exit_keywords:
        break
    else:
        my_config = {'configurable': {'thread_id': thread_id}}
        initial_state = {'messages': HumanMessage(content=user_msg)}
        output = workflow.invoke(initial_state, config=my_config)

        print(output['messages'][-1].content)

The result of adding 2 and 4 is 6.
The current Prime Minister of India is Narendra Modi.
